# 04: Model Evaluation

## Project: AgroVision
**Purpose:** Assess the final model's performance on the unseen **test set**. Generate industry-standard metrics (Confusion Matrix, F1-Score) to certify the model for deployment.

### Key Objectives
1.  **Model Loading:** Retrieve the production-ready model artifact directly from the **MLflow Model Registry** to ensure consistency with the tracked experiment.
2.  **Test Set Evaluation:** Calculate the quantitative accuracy and loss on the held-out test dataset.
3.  **Error Analysis:** Visualize the **Confusion Matrix** to identify specific classes where the model struggles (e.g., confusing "stale_banana" with "fresh_banana").
4.  **Inference Visualization:** Display model predictions on random test samples to qualitatively verify performance.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

## 1. Environment, Configuration & Model Loading
Define project paths, configure the compute device, and load the full model artifact directly from **MLflow** to ensure architecture consistency for inference.

In [ ]:
PROJECT_ROOT_DIR_PATH = Path.cwd().parent

DATA_DIR_PATH = PROJECT_ROOT_DIR_PATH / "data"
RAW_DATA_DIR_PATH = DATA_DIR_PATH / "raw"
PROCESSED_DATA_DIR_PATH = DATA_DIR_PATH / "processed"
DATASET_CLEAN_FILE_PATH = PROCESSED_DATA_DIR_PATH / "dataset_clean.csv"

ARTIFACTS_DIR_PATH = PROJECT_ROOT_DIR_PATH / "artifacts"
NORMALIZATION_STATS_FILE_PATH = ARTIFACTS_DIR_PATH / "normalization_stats.json"
MODELS_DIR_PATH = PROJECT_ROOT_DIR_PATH / "models"

MLFLOW_SERVER_URI = "http://0.0.0.0:5000"
MLFLOW_MODEL_URI = "models:/model_fruit_freshness@champion"

mlflow.set_tracking_uri(MLFLOW_SERVER_URI)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = mlflow.pytorch.load_model(MLFLOW_MODEL_URI, map_location=DEVICE)
model.eval()

print(f"Device: {DEVICE}")

## 2. Load Metadata & Statistics
Import the processed dataset index and the normalization statistics computed during the EDA phase.

In [ ]:
df = pd.read_csv(DATASET_CLEAN_FILE_PATH)

classes = sorted(df["label"].unique())
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
idx_to_class = {i: cls_name for i, cls_name in enumerate(classes)}

with open(NORMALIZATION_STATS_FILE_PATH) as f:
    stats = json.load(f)
    mean = stats["mean"]
    std = stats["std"]

print(f"Classes: {classes}")
print(f"Normalization: mean={mean}, std={std}")

## 3. Custom Dataset Definition
Implement the `AgroVisionDataset` class to handle image loading and label encoding.

In [ ]:
class AgroVisionDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.root_dir / row["filepath"]

        image = Image.open(img_path).convert("RGB")
        label_str = row["label"]
        label = class_to_idx[label_str]

        if self.transform:
            image = self.transform(image)

        return image, label

## 4. Transforms & Test Loader
Prepare the inference pipeline using **deterministic transformations** (`Resize` + `CenterCrop`) to ensure reproducible evaluation.

In [ ]:
test_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)

test_df = df[df["split"] == "test"]

test_dataset = AgroVisionDataset(test_df, RAW_DATA_DIR_PATH, transform=test_transform)

BATCH_SIZE = 32
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

print(f"Test set size: {len(test_dataset)} images")

## 5. Inference Loop
Iterate through the test DataLoader using `torch.no_grad()` to collect predictions and true labels.

In [ ]:
y_true = []
y_pred = []

print("Starting inference...")

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing"):
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

## 6. Quantitative Analysis
Evaluate the model using the classification report (precision, recall, f1-score) and confusion matrix.

In [ ]:
print(classification_report(y_true, y_pred, target_names=classes))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix: Test Set")
plt.show()

## 7. Qualitative Analysis
Visualize specific misclassifications to identify failure modes (e.g., similar rot patterns or image quality issues).

In [ ]:
def denormalize(tensor, mean, std):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return tensor * std + mean


y_true_arr = np.array(y_true)
y_pred_arr = np.array(y_pred)
error_indices = np.where(y_true_arr != y_pred_arr)[0]

print(f"Total misclassified images: {len(error_indices)}")

num_display = min(len(error_indices), 5)

if num_display > 0:
    fig, axes = plt.subplots(1, num_display, figsize=(15, 5))

    if num_display == 1:
        axes = [axes]

    for i, idx in enumerate(error_indices[:num_display]):
        img_tensor, _ = test_dataset[idx]
        img_denorm = denormalize(img_tensor, mean, std)
        img_display = transforms.ToPILImage()(img_denorm)

        true_lbl = idx_to_class[y_true_arr[idx]]
        pred_lbl = idx_to_class[y_pred_arr[idx]]

        axes[i].imshow(img_display)
        axes[i].set_title(f"True: {true_lbl}\nPred: {pred_lbl}")
        axes[i].axis("off")

    plt.tight_layout()

## 8. Summary & Transition to Production
The evaluation phase is complete. The model has been audited on the test set, providing a clear assessment of performance and potential failure modes.

**Key Outcomes:**
1.  **Inference:** Validated predictions were generated for the unseen test set.
2.  **Metrics:** The Confusion Matrix and Classification Report were produced for the best-performing model artifacts.
3.  **Auditing:** Specific errors were visualized to identify semantic confusion or data quality issues.

**Next Steps:**
1.  **Modularization:** The experimental "lab" phase (notebooks) is concluded. Development shifts to the production codebase within the `src/` directory.
2.  **Inference Logic:** Preprocessing and model loading logic will be encapsulated into a dedicated service (e.g., `freshness_service.py`) to support the API layer.
3.  **API Integration:** The trained model will be exposed through the `freshness_controller.py` to handle real-time image classification requests.